In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft loralib

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-3w0xxzo1/unsloth_35ed1bf717824859b67f84b594cda53f
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-3w0xxzo1/unsloth_35ed1bf717824859b67f84b594cda53f
  Resolved https://github.com/unslothai/unsloth.git to commit 9c2eacc35e3f5f3f33af3b2c3cad42a8dd4c5ec2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 19.5 MB/s eta 0:00:00
   ━

In [12]:
import os
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

In [4]:
max_seq_length = 2048  # Optimized for T4 memory constraints
dtype = None           # Auto-detection (Float16 for T4)
load_in_4bit = True    # 4bit quantization to save VRAM

In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,               # Rank parameter (higher = more capacity, more VRAM usage)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,     # Optimized at 0 for Unsloth speedups
    bias="none",
    use_gradient_checkpointing="unsloth", # Crucial for 16GB VRAM efficiency
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

Unsloth 2026.6.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [7]:
chatml_prompt = """<|im_start|>system
You are a helpful, precise, and articulate AI assistant.
<|im_end|>
<|im_start|>user
{}
<|im_end|>
<|im_start|>assistant
{}"""

EOS_TOKEN = tokenizer.eos_token

In [8]:
def format_alpaca_to_chatml(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        if input_text and input_text.strip() != "":
            full_prompt = f"{instruction}\nContext: {input_text}"
        else:
            full_prompt = instruction

        # Format matching the explicit system/user/assistant breakdown
        formatted_text = chatml_prompt.format(full_prompt, output) + EOS_TOKEN
        texts.append(formatted_text)
    return { "text" : texts }



In [9]:
# Load text-generation dataset and map over it
dataset = load_dataset("yahma/alpaca-cleaned", split="train")
dataset = dataset.map(format_alpaca_to_chatml, batched=True)

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [13]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=SFTConfig(                                   # <-- FIX: Replaced TrainingArguments
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        save_strategy="no",                           # <-- FIX: Bypasses the PicklingError
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/51760 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [14]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 51,760 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Step,Training Loss
1,0.838743
2,0.782292
3,0.950333
4,1.045218
5,1.022760
6,0.950448
7,0.719386
8,1.038938
9,0.982192
10,0.905510


In [15]:
# Create a dedicated directory inside your Google Drive
save_directory = "/content/drive/MyDrive/llm_adapters/text_gen_lora_adapter"

# Save both adapter configurations and weights
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

print(f"Adapters safely secured in Google Drive at: {save_directory}")

print(f"Training completed successfully. Peak VRAM used: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/llm_adapters/text_gen_lora_adapter/tokenizer_config.json.


Adapters safely secured in Google Drive at: /content/drive/MyDrive/llm_adapters/text_gen_lora_adapter
Training completed successfully. Peak VRAM used: 6.54 GB
